In [1]:
# Lets create a table with name 'images' in Qdrant. We shall add embedding for each image here.
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
from qdrant_client.http.models import HnswConfigDiff

COL = 'images'
client = QdrantClient("http://localhost:6333")
if client.collection_exists(collection_name=COL):
    client.delete_collection(collection_name=COL)

if not client.collection_exists(collection_name=COL):
    client.create_collection(
        collection_name=COL,
        vectors_config=VectorParams(size=512, distance=Distance.COSINE),
        hnsw_config=HnswConfigDiff(m=16, ef_construct=200),
        optimizers_config={"max_segment_size": 100000}
        # optional params:
        # shards=1,
        # replication_factor=1,
    )

In [2]:
def dummy_run(config, bindings, output_buffer, timeout_ms = 1000,):
    config.run([bindings], timeout_ms)
    vec = bindings.output().get_buffer()
    return vec


In [3]:
from PIL import Image
# set input output buffer, run model and return embedding vec
def process_image(base_folder, rel_path, config, bindings, output_buffer, timeout_ms = 1000):
    try:
        # Resize image and 
        image_path = base_folder / rel_path
        image = Image.open(image_path).convert("RGB").resize((224, 224))
        input_buffer = np.asarray(image).astype(np.uint8)  # uint8, no batch dimension

        # IO Binding
        bindings.input().set_buffer(input_buffer)
        bindings.output().set_buffer(output_buffer)
        
        # Run model synchronous inference and access the output buffers
        
        return dummy_run(config, bindings, output_buffer)
    except Exception as e:
        return None

In [4]:
from pathlib import Path
import time
from qdrant_client.models import PointStruct
import numpy as np
import uuid 
profile_batch_size = 100
max_images = 5000

def process_image_dir(base_folder:Path, relative_path, config, bindings, output_buffer, vec_table):
    path:Path = base_folder / relative_path
   
    if path.is_dir():
        total_time = 0.0
        success_count = 0
        for ext in ("*.jpg",):  # , "*.jpeg", "*.png"
            all_files = list(path.rglob(ext))
            files_2_process = all_files #[:max_images]
            total = len(files_2_process)
            for i, image_path in enumerate(files_2_process):
                start = time.perf_counter()
                #print(f"processing {i+1}/{total}")
                rel_path = (image_path.resolve()).relative_to(base_folder)
                vec = process_image(base_folder, rel_path, config, bindings, output_buffer)
               
                if vec is not None:
                    vec_f32 = vec.astype("float32")
                    vec_f32 /= np.linalg.norm(vec_f32)
                    client.upsert(
                        collection_name="images",
                        points=[PointStruct(
                        id=str(uuid.uuid4()),
                        vector=vec_f32,
                        payload={"filename": rel_path}
                    )]
                    )
                    end = time.perf_counter()
                    success_count = success_count + 1
                    total_time = total_time + (end - start)
                    if (success_count  % profile_batch_size) == 0:
                        print(f"processed {i+1} / {total},   average: {(total_time * 1000)/success_count:.4f} msec" )    
    else:
        print(f"processing {relative_path}")
        vec = process_image(base_folder, relative_path, config, bindings, output_buffer)
        if vec:
            # vec_table.add([{ "id": str(relative_path), "vector": vec }])
            vec_f32 = vec.astype("float32")
            vec_f32 /= np.linalg.norm(vec_f32)
            client.upsert(
                        collection_name="images",
                        points=[PointStruct(
                        id=str(uuid.uuid4()),
                        vector=vec_f32,
                        payload={"filename": relative_path}
                    )]
                    )
        else:
            print(f"failed {relative_path}")

In [5]:
import numpy as np
from hailo_platform import VDevice, HailoSchedulingAlgorithm
# Create a inference mode, configure, create binding and then invoke process_image_dir
def process(hef, base_folder, relative_path, vec_table):
    params = VDevice.create_params()
    params.scheduling_algorithm = HailoSchedulingAlgorithm.ROUND_ROBIN
    with VDevice(params) as vdevice:
        # Create an infer model from an HEF:
        infer_model = vdevice.create_infer_model(hef)

        # Configure the infer model and create bindings for it
        with infer_model.configure() as config:
            bindings = config.create_bindings()
            output_buffer = np.empty(list(infer_model.output().shape), dtype=np.uint8)
            process_image_dir(base_folder, relative_path,  config, bindings, output_buffer, vec_table)

In [ ]:
import time

DB_PATH = "image_embeddings2.lance"
# Setting up paths.
image_folder = Path("/home/anandas/test_images/Datewise/")
base_folder = Path("/home/anandas/test_images")
MODEL_HEF = "resnet_v1_18_feature.hef"

start = time.perf_counter()
# table = create_table(DB_PATH)
process(hef=MODEL_HEF, base_folder=base_folder, relative_path=Path('Datewise'), vec_table="")
end = time.perf_counter()

print(f"Time taken: {end - start:.4f} seconds")
